In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

2025-04-30 13:43:42.118334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746020622.354540      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746020622.465837      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
dataset = load_dataset("wmt14", "fr-en") #English-to-French

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

train-00001-of-00030.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

train-00002-of-00030.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/536k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40836715 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [3]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 40836715
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3003
    })
})
{'translation': {'en': 'Resumption of the session', 'fr': 'Reprise de la session'}}


In [4]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [5]:
small_train_dataset = dataset["train"].select(range(10000))
small_valid_dataset = dataset["validation"].select(range(1000))

In [6]:
def preprocess_function(examples):
    inputs = ["translate English to French: " + ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length") #nbr
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
tokenized_train_dataset = small_train_dataset.map(preprocess_function, batched=True)
tokenized_valid_dataset = small_valid_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["translation"])
tokenized_valid_dataset = tokenized_valid_dataset.remove_columns(["translation"])

In [9]:
print(tokenized_train_dataset[0])

{'input_ids': [13959, 1566, 12, 2379, 10, 419, 4078, 102, 1575, 13, 8, 2363, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [7144, 7854, 20, 50, 2363, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [10]:
tokenized_train_dataset.save_to_disk("/kaggle/working/tokenized_train")
tokenized_valid_dataset.save_to_disk("/kaggle/working/tokenized_valid")

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model) #pads to the longest sequence in that batch

In [13]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

In [15]:
import os
# Disable W&B logging
os.environ["WANDB_MODE"] = "disabled"

In [16]:
!pip install evaluate


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==1

In [17]:
import evaluate

bleu_metric = evaluate.load("bleu")


In [18]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Debug: Print shapes and sample values
    print(f"Predictions shape: {predictions.shape}, Labels shape: {labels.shape}")
    print(f"Sample prediction: {predictions[0][:10]}, Sample label: {labels[0][:10]}")
    
    # Sanitize predictions: Clip to valid token ID range [0, tokenizer.vocab_size)
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    # Handle negative label IDs (used for padding in T5)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Decode predictions and labels
    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f"Decoding error: {e}")
        return {"bleu": 0.0}
    
    # Filter out empty or invalid predictions/labels
    paired_results = [(pred, label) for pred, label in zip(decoded_preds, decoded_labels) if pred.strip() and label.strip()]
    if not paired_results:
        print("Warning: No valid predictions or labels for BLEU computation")
        return {"bleu": 0.0}
    
    decoded_preds, decoded_labels = zip(*paired_results)
    # Prepare for BLEU
    decoded_preds = [pred.split() for pred in decoded_preds]
    decoded_labels = [[label.split()] for label in decoded_labels]
    
    # Compute BLEU
    try:
        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        print(f"BLEU score: {result['bleu']}")
        return {"bleu": result["bleu"]}
    except Exception as e:
        print(f"BLEU computation error: {e}")
        return {"bleu": 0.0}


In [19]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    fp16=True,  
    logging_dir="/kaggle/working/logs",
    logging_steps=500,
    report_to="none",
)

In [20]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [22]:
import time 

start_time = time.time()

# Train the model
trainer.train()

# Calculate and log total training time
total_training_time = time.time() - start_time
total_training_time_min = total_training_time / 60
print(f"Total training time: {total_training_time_min:.2f} minutes")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Bleu
1,No log,0.296537,0.000000
2,1.604000,0.290136,0.000000
3,1.604000,0.289714,0.000000


Predictions shape: (1000, 128), Labels shape: (1000, 128)
Sample prediction: [0 0 0 0 0 0 0 0 0 0], Sample label: [ 2435 17295  1417 15727  7043   171  3386    52    50  1417]
BLEU computation error: Predictions and/or references don't match the expected format.
Expected format:
Feature option 0: {'predictions': Value(dtype='string', id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='sequence'), length=-1, id='references')}
Feature option 1: {'predictions': Value(dtype='string', id='sequence'), 'references': Value(dtype='string', id='sequence')},
Input predictions: ['Les', 'dirigeants', 'républicains', ..., 'la', 'fraude', 'électorale.'],
Input references: [['Les', 'dirigeants', 'républicains', 'justifièrent', 'leur', 'politique', 'par', 'la', 'nécessité', 'de', 'lutter', 'contre', 'la', 'fraude', 'électorale.']]


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Predictions shape: (1000, 128), Labels shape: (1000, 128)
Sample prediction: [    0  2435 17295  1417 15727  7043   171  3386    52    50], Sample label: [ 2435 17295  1417 15727  7043   171  3386    52    50  1417]
BLEU computation error: Predictions and/or references don't match the expected format.
Expected format:
Feature option 0: {'predictions': Value(dtype='string', id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='sequence'), length=-1, id='references')}
Feature option 1: {'predictions': Value(dtype='string', id='sequence'), 'references': Value(dtype='string', id='sequence')},
Input predictions: ['Une', 'stratégie', 'républicaine', ..., 'la', 'réélection', "d'Obama"],
Input references: [['Une', 'stratégie', 'républicaine', 'pour', 'contrer', 'la', 'réélection', "d'Obama"]]


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Predictions shape: (1000, 128), Labels shape: (1000, 128)
Sample prediction: [    0  2435 17295  1417 15727  7043   171  3386    52    50], Sample label: [ 2435 17295  1417 15727  7043   171  3386    52    50  1417]
BLEU computation error: Predictions and/or references don't match the expected format.
Expected format:
Feature option 0: {'predictions': Value(dtype='string', id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='sequence'), length=-1, id='references')}
Feature option 1: {'predictions': Value(dtype='string', id='sequence'), 'references': Value(dtype='string', id='sequence')},
Input predictions: ['Une', 'stratégie', 'républicaine', ..., 'la', 'réélection', "d'Obama"],
Input references: [['Une', 'stratégie', 'républicaine', 'pour', 'contrer', 'la', 'réélection', "d'Obama"]]
Total training time: 11.06 minutes


In [23]:
# Save the model and tokenizer
model.save_pretrained("/kaggle/working/trained_model")
tokenizer.save_pretrained("/kaggle/working/trained_model")


('/kaggle/working/trained_model/tokenizer_config.json',
 '/kaggle/working/trained_model/special_tokens_map.json',
 '/kaggle/working/trained_model/spiece.model',
 '/kaggle/working/trained_model/added_tokens.json',
 '/kaggle/working/trained_model/tokenizer.json')

In [56]:
from transformers import pipeline
translator = pipeline("translation_en_to_fr", model="/kaggle/working/trained_model", tokenizer="/kaggle/working/trained_model")
test_sentence = "Hello , my name is Ikram"
result = translator(test_sentence)
print(result)

Device set to use cuda:0


[{'translation_text': 'Bonjour , mon nom est Ikram'}]


In [24]:
from transformers import pipeline
translator = pipeline("translation_en_to_fr", model="/kaggle/working/trained_model", tokenizer="/kaggle/working/trained_model")
test_sentence = "Hello , my name is Ikram"
result = translator(test_sentence)
print(result)

Device set to use cuda:0


[{'translation_text': 'Bonjour , mon nom est Ikram'}]


In [26]:
import json
from transformers import pipeline

# Initialize the translation pipeline
try:
    translator = pipeline("translation_en_to_fr", model="/kaggle/working/trained_model", tokenizer="/kaggle/working/trained_model")
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    exit()

# List of test sentences
test_sentences = [
    "Hello, my name is Ikram",
    "How are you today?",
    "I love to learn new languages.",
    "The sun is shining brightly.",
    "Can you help me with my homework?",
    "This is a beautiful day.",
    "What time is it?",
    "I am going to the park.",
    "Thank you for your help.",
    "My favorite color is blue."
]

# Store results
results = []

# Test each sentence
for sentence in test_sentences:
    try:
        result = translator(sentence)
        translated_text = result[0]["translation_text"]
        print(f"Input: {sentence}")
        print(f"Output: {translated_text}\n")
        results.append({"input": sentence, "output": translated_text})
    except Exception as e:
        print(f"Error translating '{sentence}': {e}")
        results.append({"input": sentence, "output": "Error"})

# Save results to a file
with open("/kaggle/working/translation_results.json", "w") as f:
    json.dump(results, f, indent=4)
print("Translation results saved to /kaggle/working/translation_results.json")

Device set to use cuda:0


Input: Hello, my name is Ikram
Output: Bonjour, mon nom est Ikram

Input: How are you today?
Output: Comment êtes-vous aujourd'hui ?

Input: I love to learn new languages.
Output: J'aime apprendre de nouvelles langues.

Input: The sun is shining brightly.
Output: Le soleil brille.

Input: Can you help me with my homework?
Output: Pouvez-vous m'aider à faire mes devoirs?

Input: This is a beautiful day.
Output: C'est une belle journée.

Input: What time is it?
Output:  quel moment ?

Input: I am going to the park.
Output: Je me rends au parc.

Input: Thank you for your help.
Output: Je vous remercie de votre aide.

Input: My favorite color is blue.
Output: Ma couleur préférée est bleue.

Translation results saved to /kaggle/working/translation_results.json
